In [0]:
from pyspark.sql import functions as F

b = spark.table("workspace.rc_bronze.survey_results").where("_survey_year = 2025")
src_cols = [r.bronze_name for r in spark.table("workspace.rc_bronze.column_map")
            .where("survey_year = 2025").orderBy("ordinal").collect()]
content_cols = [c for c in src_cols if c != "ResponseId"]
n = b.count()

# Doublons de contenu (toutes colonnes sauf l'identifiant)
groups = b.groupBy(*content_cols).count().where("count > 1")
n_groups = groups.count()
n_rows_in_groups = groups.agg(F.sum("count")).collect()[0][0] or 0
print(f"Groupes de lignes au contenu identique : {n_groups:,}")
print(f"Lignes concernées : {n_rows_in_groups:,} | lignes en excès : {n_rows_in_groups - n_groups:,}")

# Nombre de colonnes renseignées (≠ 'NA') par répondant
answered = sum(F.when(F.col(c) != "NA", 1).otherwise(0) for c in content_cols)
b2 = b.withColumn("n_answered", answered)
b2.agg(F.min("n_answered").alias("min"),
       F.expr("percentile_approx(n_answered, array(0.05, 0.25, 0.5, 0.75, 0.95))").alias("quantiles"),
       F.max("n_answered").alias("max")).show(truncate=False)
print("Répondants avec 5 réponses ou moins :", b2.where("n_answered <= 5").count())

Groupes de lignes au contenu identique : 300
Lignes concernées : 1,847 | lignes en excès : 1,547
+---+---------------------+---+
|min|quantiles            |max|
+---+---------------------+---+
|2  |[6, 36, 99, 120, 134]|156|
+---+---------------------+---+

Répondants avec 5 réponses ou moins : 2015


In [0]:
semi = b.agg(*[F.sum(F.col(c).contains(";").cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
multi = sorted([(c, v) for c, v in semi.items() if v and v > 0], key=lambda x: -x[1])
print(f"Colonnes contenant ';' : {len(multi)}")
for c, v in multi:
    print(f"  {c}: {v:,} lignes ({v / n * 100:.1f} %)")

b.where("DevType LIKE '%;%'").select("DevType").show(3, truncate=False)

Colonnes contenant ';' : 78
  LearnCode: 30,901 lignes (62.8 %)
  LanguageHaveWorkedWith: 30,513 lignes (62.0 %)
  CommPlatformHaveWorkedWith: 28,328 lignes (57.6 %)
  AIHuman: 26,274 lignes (53.4 %)
  OfficeStackAsyncHaveWorkedWith: 26,047 lignes (53.0 %)
  LanguageWantToWorkWith: 24,334 lignes (49.5 %)
  PlatformHaveWorkedWith: 22,776 lignes (46.3 %)
  AIToolDon_t_plan_to_use_AI_for_this_task: 22,358 lignes (45.5 %)
  DevEnvsHaveWorkedWith: 22,310 lignes (45.4 %)
  CommPlatformWantToWorkWith: 21,182 lignes (43.1 %)
  AILearnHow: 20,929 lignes (42.5 %)
  OpSysPersonal_use: 20,926 lignes (42.5 %)
  LanguageAdmired: 20,879 lignes (42.4 %)
  DatabaseHaveWorkedWith: 20,823 lignes (42.3 %)
  CommPlatformAdmired: 20,723 lignes (42.1 %)
  WebframeHaveWorkedWith: 19,174 lignes (39.0 %)
  OfficeStackAsyncWantToWorkWith: 18,440 lignes (37.5 %)
  AIToolPlan_to_partially_use_AI: 18,351 lignes (37.3 %)
  AIToolCurrently_partially_AI: 18,096 lignes (36.8 %)
  AIFrustration: 17,956 lignes (36.5 %)
 

In [0]:
from pyspark.sql import functions as F

# --- Préparation (autonome) ---
b = spark.table("workspace.rc_bronze.survey_results").where("_survey_year = 2025")
src_cols = [r.bronze_name for r in spark.table("workspace.rc_bronze.column_map")
            .where("survey_year = 2025").orderBy("ordinal").collect()]
content_cols = [c for c in src_cols if c != "ResponseId"]
answered = sum(F.when(F.col(c) != "NA", 1).otherwise(0) for c in content_cols)
b2 = b.withColumn("n_answered", answered)

# --- 1. Où sont les doublons de contenu ? ---
print("=== 1. Doublons de contenu, par nombre de réponses ===")
rh = b2.withColumn("row_hash", F.sha2(F.concat_ws("\u0001", *content_cols), 256))
g = rh.groupBy("row_hash").agg(F.count("*").alias("grp_size"), F.max("n_answered").alias("n_answered"))
(g.where("grp_size > 1")
  .withColumn("tranche", F.when(F.col("n_answered") <= 5, "a) <= 5 réponses")
                          .when(F.col("n_answered") <= 20, "b) 6 à 20")
                          .otherwise("c) > 20"))
  .groupBy("tranche")
  .agg(F.count("*").alias("groupes"), F.sum(F.col("grp_size") - 1).alias("lignes_en_exces"))
  .orderBy("tranche").show(truncate=False))

print("Lignes quasi vides (<= 5 réponses) par MainBranch :")
b2.where("n_answered <= 5").groupBy("MainBranch").count().orderBy(F.desc("count")).show(truncate=False)

# --- 2. Qui a répondu à quoi ? ---
print("=== 2. Populations par MainBranch ===")
(b.groupBy("MainBranch")
  .agg(F.count("*").alias("n"),
       F.sum((F.col("AISelect") != "NA").cast("int")).alias("AISelect_repondu"),
       F.sum((F.col("AIAcc") != "NA").cast("int")).alias("AIAcc_repondu"),
       F.sum((F.col("JobSat") != "NA").cast("int")).alias("JobSat_repondu"),
       F.sum((F.col("ConvertedCompYearly") != "NA").cast("int")).alias("Salaire_renseigne"))
  .orderBy(F.desc("n")).show(truncate=False))
b.groupBy("Age").count().orderBy(F.desc("count")).show(truncate=False)

# --- 3. Les libellés des réponses multiples sont-ils propres ? ---
print("=== 3. Libellés des réponses multiples ===")
for c in ["LearnCode", "AIFrustration", "AIToolCurrently_partially_AI", "AIAgentChallengesStrongly_agree"]:
    base = b.where(F.col(c) != "NA")
    ch = base.select(F.explode(F.split(c, ";")).alias("choix"))
    print(f"\n----- {c} : {base.count():,} répondants, "
          f"{ch.where(F.col('choix') != F.trim(F.col('choix'))).count()} choix avec espaces en trop")
    ch.groupBy("choix").count().orderBy(F.desc("count")).show(15, truncate=False)

# --- 4. Salaires ---
print("=== 4. Salaires ===")
print("Colonnes liées à la rémunération :", [c for c in src_cols if any(k in c for k in ["Comp", "Currency", "Salary"])])
sal = b.select(F.expr("try_cast(ConvertedCompYearly as double)").alias("comp"),
               (F.col("ConvertedCompYearly") != "NA").alias("declared"))
print("Valeurs renseignées mais non numériques :", sal.where("declared and comp is null").count())
(sal.where("comp is not null")
    .agg(F.count("*").alias("n"), F.min("comp").alias("min"),
         F.expr("percentile_approx(comp, array(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99))").alias("quantiles"),
         F.max("comp").alias("max"),
         F.sum((F.col("comp") <= 0).cast("int")).alias("n_non_positifs"),
         F.sum((F.col("comp") > 1000000).cast("int")).alias("n_sup_1M"))
    .show(truncate=False))

=== 1. Doublons de contenu, par nombre de réponses ===
+----------------+-------+---------------+
|tranche         |groupes|lignes_en_exces|
+----------------+-------+---------------+
|a) <= 5 réponses|182    |1346           |
|b) 6 à 20       |118    |201            |
+----------------+-------+---------------+

Lignes quasi vides (<= 5 réponses) par MainBranch :
+------------------------------------------------------------------------------------------+-----+
|MainBranch                                                                                |count|
+------------------------------------------------------------------------------------------+-----+
|I am a developer by profession                                                            |1284 |
|I am learning to code                                                                     |230  |
|I am not primarily a developer, but I write code sometimes as part of my work/studies     |206  |
|I code primarily as a hobby            

In [0]:
from pyspark.sql import functions as F
from functools import reduce

b = spark.table("workspace.rc_bronze.survey_results").where("_survey_year = 2025")
src_cols = [r.bronze_name for r in spark.table("workspace.rc_bronze.column_map")
            .where("survey_year = 2025").orderBy("ordinal").collect()]

# A. Structure des matrices : une paire (répondant, élément) apparaît-elle à plusieurs niveaux ?
print("=== A. Matrices ===")
for prefix in ["AITool", "AIAgentImpact", "AIAgentChallenges"]:
    lvl = [c for c in src_cols if c.startswith(prefix)]
    parts = [b.where(F.col(c) != "NA")
              .select("ResponseId", F.lit(c).alias("lvl"), F.explode(F.split(c, ";")).alias("item"))
             for c in lvl]
    d = reduce(lambda x, y: x.unionByName(y), parts)
    dup = d.groupBy("ResponseId", "item").count().where("count > 1").count()
    print(f"{prefix} : {len(lvl)} colonnes {lvl}")
    print(f"   répondants avec au moins une colonne renseignée : {d.select('ResponseId').distinct().count():,}"
          f" | paires (répondant, élément) : {d.count():,} | paires présentes à plusieurs niveaux : {dup:,}")

# B. Colonnes à typer : numériques ou textuelles ?
print("\n=== B. YearsCode / WorkExp ===")
for c in ["WorkExp", "YearsCode"]:
    x = b.where(F.col(c) != "NA").withColumn("num", F.expr(f"try_cast({c} as double)"))
    print(f"--- {c} : numériques {x.where('num is not null').count():,} | non numériques {x.where('num is null').count():,}")
    x.where("num is null").groupBy(c).count().orderBy(F.desc("count")).show(8, truncate=False)
    x.agg(F.min("num").alias("min"), F.max("num").alias("max")).show()

print("=== C. Valeurs des colonnes ordinales ===")
for c in ["JobSat", "AIThreat", "AIComplex", "OrgSize", "RemoteWork", "ICorPM"]:
    print(f"--- {c}")
    b.where(F.col(c) != "NA").groupBy(c).count().orderBy(F.desc("count")).show(15, truncate=False)

print("=== D. Pays ===")
print("Pays distincts :", b.where("Country != 'NA'").select("Country").distinct().count())
b.where("Country != 'NA'").groupBy("Country").count().orderBy(F.desc("count")).show(8, truncate=False)

# E. Salaire : définition et bornes
print("=== E. Salaire ===")
b.where("Currency != 'NA'").groupBy("Currency").count().orderBy(F.desc("count")).show(5, truncate=False)
usd = (b.where(F.col("Currency").startswith("USD"))
        .select(F.expr("try_cast(CompTotal as double)").alias("ct"),
                F.expr("try_cast(ConvertedCompYearly as double)").alias("cc"))
        .where("ct is not null and cc is not null"))
usd.agg(F.count("*").alias("lignes_USD"), F.sum((F.col("ct") == F.col("cc")).cast("int")).alias("CompTotal_egal_Converted")).show()
comp = b.select(F.expr("try_cast(ConvertedCompYearly as double)").alias("comp")).where("comp is not null")
comp.agg(F.sum((F.col("comp") < 1000).cast("int")).alias("moins_de_1000"),
         F.sum((F.col("comp") > 1e6).cast("int")).alias("plus_de_1M"),
         F.expr("percentile_approx(comp, 0.5)").alias("mediane_toutes_valeurs")).show()
comp.where("comp between 1000 and 1000000").agg(
    F.count("*").alias("n_dans_bornes"), F.expr("percentile_approx(comp, 0.5)").alias("mediane_dans_bornes")).show()

=== A. Matrices ===
AITool : 5 colonnes ['AIToolCurrently_partially_AI', 'AIToolDon_t_plan_to_use_AI_for_this_task', 'AIToolPlan_to_partially_use_AI', 'AIToolPlan_to_mostly_use_AI', 'AIToolCurrently_mostly_AI']
   répondants avec au moins une colonne renseignée : 31,066 | paires (répondant, élément) : 394,221 | paires présentes à plusieurs niveaux : 0
AIAgentImpact : 5 colonnes ['AIAgentImpactSomewhat_agree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat_disagree', 'AIAgentImpactStrongly_agree', 'AIAgentImpactStrongly_disagree']
   répondants avec au moins une colonne renseignée : 12,840 | paires (répondant, élément) : 90,272 | paires présentes à plusieurs niveaux : 0
AIAgentChallenges : 5 colonnes ['AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat_disagree', 'AIAgentChallengesStrongly_agree', 'AIAgentChallengesSomewhat_agree', 'AIAgentChallengesStrongly_disagree']
   répondants avec au moins une colonne renseignée : 28,954 | paires (répondant, élément) : 174,136 | paires présent